In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score


data_root = r"C:\Users\jontr\OneDrive\Documents\ECS 111\ECS 111 Final Project\data\splits"
rotations = ["rotation_0", "rotation_1", "rotation_2", "rotation_3"]

# list to hold the final results
all_results = []

In [2]:
# for loop that goes through each of the different generator rotations, each one leaving out one generater for the training dataset
for rot in rotations:
    # loading training, in-distribution test, and cross-generator test datasets for the current rotation
    path = os.path.join(data_root, rot)
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_in_df = pd.read_csv(os.path.join(path, "test_indist.csv"))
    test_cross_df = pd.read_csv(os.path.join(path, "test_crossgen.csv"))
    

    # use 'text' for the content and 'label' for the target
    train_text = train_df['text'].astype(str).fillna("")
    test_in_text = test_in_df['text'].astype(str).fillna("")
    test_cross_text = test_cross_df['text'].astype(str).fillna("")
    
    y_train = train_df['label']
    y_test_in = test_in_df['label']
    y_test_cross = test_cross_df['label']

    # vectorizing the content using TF-IDF
    # ngram_range=(1, 2) helps detect AI-specific phrases with bigrams
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
    x_train = vectorizer.fit_transform(train_text)
    x_test_in = vectorizer.transform(test_in_text)
    x_test_cross = vectorizer.transform(test_cross_text)


   # training Logistic Regression with random state to keep reproducability
    model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
    model.fit(x_train, y_train)
    # getting evauluation scores such as the F1 and AUC for the in-distrubution test set
    prob_in = model.predict_proba(x_test_in)[:, 1]  # model gives probabilites for each target clas
    pred_in = (prob_in >= 0.5).astype(int)  # thresholding at 0.5 to get binary predictions
    f1_in = f1_score(y_test_in, pred_in)
    auc_in = roc_auc_score(y_test_in, prob_in)

    # getting evaluation scores such as F1 and AUC for the cross-generator test set
    prob_cross = model.predict_proba(x_test_cross)[:, 1]  # models give probabilities for each target class
    pred_cross = (prob_cross >= 0.5).astype(int)  # thresholding at 0.5 to get binary predictions
    f1_cross = f1_score(y_test_cross, pred_cross)
    auc_cross = roc_auc_score(y_test_cross, prob_cross)

    # checking if the model performs better on the in-distribution test set than the cross-generator test set, and calculating the gap
    test_cross_df['preds'] = pred_cross
    gen_scores = {}
    for gen in test_cross_df['generator'].unique():
        subset = test_cross_df[test_cross_df['generator'] == gen]
        # only score if the generator is an AI (label 1)
        if subset['label'].iloc[0] == 1:
            gen_f1 = f1_score(subset['label'], subset['preds'])
            gen_scores[f"{gen}_F1"] = gen_f1

    # storing the results into a dictionary and appending it to the list of all results
    res = {
        "Rotation": rot,
        "In-Dist F1": f1_in,
        "In-Dist AUC": auc_in,
        "Cross-Gen F1": f1_cross,
        "Cross-Gen AUC": auc_cross,
        "Gap (F1)": f1_in - f1_cross
    }
    res.update(gen_scores) # adding specific generator scores
    all_results.append(res)
    
    print(f" {rot} Complete | Gap: {(f1_in - f1_cross):.4f}")

 rotation_0 Complete | Gap: 0.0102
 rotation_1 Complete | Gap: 0.0151
 rotation_2 Complete | Gap: 0.0379
 rotation_3 Complete | Gap: -0.0026


In [3]:
# converting list of results into a pandas dataframe
results_df = pd.DataFrame(all_results)

# calculating the averages row (Headline Numbers)
averages = results_df.mean(numeric_only=True).to_dict()
averages["Rotation"] = "AVERAGE"
summary_table = pd.concat([results_df, pd.DataFrame([averages])], ignore_index=True)

# save for the evaluation notebook
summary_table.to_csv("tfidf_baseline_results.csv", index=False)

print("\n--- TF-IDF + LOGREG BASELINE SUMMARY ---")
display(summary_table.round(4))


--- TF-IDF + LOGREG BASELINE SUMMARY ---


,Rotation,In-Dist F1,In-Dist AUC,Cross-Gen F1,Cross-Gen AUC,Gap (F1),gpt5mini_F1,deepseek_F1,gemma_F1,qwen_F1
0,rotation_0,0.9706,0.9962,0.9604,0.9951,0.0102,0.9717,NaN,NaN,NaN
1,rotation_1,0.9672,0.9964,0.9521,0.9941,0.0151,NaN,0.96,NaN,NaN
2,rotation_2,0.9738,0.9967,0.9359,0.9902,0.0379,NaN,NaN,0.9438,NaN
3,rotation_3,0.9721,0.9963,0.9747,0.9965,-0.0026,NaN,NaN,NaN,0.9856
4,AVERAGE,0.9709,0.9964,0.9558,0.9940,0.0151,0.9717,0.96,0.9438,0.9856
